In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error, mean_absolute_error

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient





def read_file(file):
    df = pd.read_csv(file)


    return df

def preprocessing(df):

    df["datetime"] = pd.to_datetime(df["datetime"])

    # Create new features
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    # Create binary weather features
    df['is_clear_weather'] = (df['weather'] == 1).astype(int)
    df['is_rainy_weather'] = (df['weather'] >= 3).astype(int)

    df['is_holiday_workingday'] = ((df['holiday'] == 1) & (df['workingday'] == 1)).astype(int)

    df.drop(columns=["datetime"], inplace=True)

    return df


def split_data(df):
    X = df.drop(columns = ["count"], axis=1)
    y = df["count"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


    return X_train, X_test, y_train, y_test


def train_model(X_train, X_test, y_train, y_test):
    model = DecisionTreeRegressor(max_depth=10, random_state=42)

    with open("data.csv", "w") as f:
        for i in X_train.iterrows():
            f.write(i)

    # mlflow starts
    with mlflow.start_run():
        mlflow.log_param("model_type", "Decision_Tree_Regressor")
        mlflow.log_param("max_depth", 10)

        # train the model
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))

        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)

        feature_importance = model.feature_importances_

        plt.figure(figsize=(10, 6))
        plt.barh(X_train.columns, feature_importance)
        plt.title("Feature Importance")
        plt.savefig("feature_importance.png")

        mlflow.log_artifact(os.path.abspath("feature_importance.png"))

        # logging the model
        # mlflow.sklearn.log_model(model, "decision_tree_model")

        print("Model Logged Successfully")
        print('=' * 50)
        print(f"RMSE: {rmse}, MAE: {mae}")
        print('=' * 50)



def hyperparam_tuning(X_train, X_test, y_train, y_test):
    model = DecisionTreeRegressor(random_state=42)

    param_grid = {
        "max_depth": [5, 10 ,15],
        "min_samples_split": [2, 10, 20]
    }

    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=param_grid,
        cv=5, scoring="neg_mean_squared_error",
        verbose=1,
        n_jobs=-1
        )
    
    with mlflow.start_run(run_name="staging") as run:
        grid_search.fit(X_train, y_train)

        best_params = grid_search.best_params_
        mlflow.log_params(best_params)
        
        best_score = -grid_search.best_score_
        mlflow.log_metric("best_cross_val_score", best_score)

        test_preds = grid_search.best_estimator_.predict(X_test)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

        mlflow.log_metric("test_RMSE", test_rmse)


        mlflow.sklearn.log_model(
                grid_search.best_estimator_,
                artifact_path="model",
                registered_model_name="BikePredictionModel"
            )








In [2]:

file = "bike-sharing-demand/train.csv"

df = read_file(file)
df = preprocessing(df)
X_train, X_test, y_train, y_test = split_data(df)
    # train_model(X_train, X_test, y_train, y_test)
    # hyperparam_tuning(X_train, X_test, y_train, y_test)


In [10]:
X_train

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,hour,day_of_week,month,is_clear_weather,is_rainy_weather,is_holiday_workingday
2930,3,0,1,1,28.70,32.575,65,12.9980,10,25,0,0,7,1,0,0
7669,2,0,1,1,22.96,26.515,52,22.0028,57,194,22,4,5,1,0,0
1346,2,0,1,1,12.30,15.910,61,6.0032,12,41,23,4,4,1,0,0
9432,3,0,0,1,23.78,27.275,60,8.9981,70,226,9,6,9,1,0,0
453,1,0,1,3,8.20,9.850,93,12.9980,1,15,23,1,2,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,1,0,0,1,6.56,8.335,47,11.0014,6,32,2,5,1,1,0,0
5191,4,0,0,1,11.48,12.880,61,19.0012,15,134,9,5,12,1,0,0
5390,4,0,0,1,11.48,13.635,48,16.9979,27,207,16,6,12,1,0,0
860,1,0,0,1,15.58,19.695,17,35.0008,6,16,7,5,2,1,0,0


In [7]:
import mlflow.pyfunc
model = mlflow.pyfunc.load_model("models:/BikePredictionModel@staging")
print(model)

2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/08 14:14:08 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/08 14:14:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/08 14:14:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.


mlflow.pyfunc.loaded_model:
  artifact_path: file:D:/Machine Learning with MLFlow/Machine-Learning-with-MLFlow/mlruns/0/models/m-d0d2f5a13b5946df86faa4f44c0d3733/artifacts
  flavor: mlflow.sklearn
  run_id: f535e227c8a94587ab6e3b33e8385510



In [20]:
[X_train.iloc[70]]

[season                     3.0000
 holiday                    0.0000
 workingday                 1.0000
 weather                    1.0000
 temp                      27.0600
 atemp                     31.0600
 humidity                  29.0000
 windspeed                  6.0032
 casual                    72.0000
 registered               256.0000
 hour                      13.0000
 day_of_week                1.0000
 month                      9.0000
 is_clear_weather           1.0000
 is_rainy_weather           0.0000
 is_holiday_workingday      0.0000
 Name: 9316, dtype: float64]

In [ ]:
[season                     3.0000
 holiday                    0.0000
 workingday                 1.0000
 weather                    1.0000
 temp                      27.0600
 atemp                     31.0600
 humidity                  29.0000
 windspeed                  6.0032
 casual                    72.0000
 registered               256.0000
 hour                      13.0000
 day_of_week                1.0000
 month                      9.0000
 is_clear_weather           1.0000
 is_rainy_weather           0.0000
 is_holiday_workingday      0.0000
]

In [25]:
[X_train.iloc[70].to_dict()]

[{'season': 3.0,
  'holiday': 0.0,
  'workingday': 1.0,
  'weather': 1.0,
  'temp': 27.06,
  'atemp': 31.06,
  'humidity': 29.0,
  'windspeed': 6.0032,
  'casual': 72.0,
  'registered': 256.0,
  'hour': 13.0,
  'day_of_week': 1.0,
  'month': 9.0,
  'is_clear_weather': 1.0,
  'is_rainy_weather': 0.0,
  'is_holiday_workingday': 0.0}]

In [24]:
model.predict([[3.0,
  0.0,
  1.0,
  1.0,
  27.06,
  31.06,
  29.0,
  6.0032,
  72.0,
  256.0,
  13.0,
  1.0,
  9.0,
  1.0,
  0.0,
  0.0]])

d:\Machine Learning with MLFlow\Machine-Learning-with-MLFlow\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


array([328.])

In [27]:
model.predict([X_train.iloc[70].to_list()])

d:\Machine Learning with MLFlow\Machine-Learning-with-MLFlow\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


array([328.])

In [31]:
pd.Series([[3.0,
  0.0,
  1.0,
  1.0,
  27.06,
  31.06,
  29.0,
  6.0032,
  72.0,
  256.0,
  13.0,
  1.0,
  9.0,
  1.0,
  0.0,
  0.0]])

0    [3.0, 0.0, 1.0, 1.0, 27.06, 31.06, 29.0, 6.003...
dtype: object

In [ ]:
{"season": 3.0,
 "holiday": 0.0,
 "workingday": 1.0,
 "weather": 1.0,
 "temp": 27.06,
 "atemp": 31.06,
 "humidity": 29.0,
 "windspeed": 6.0032,
 "casual": 72.0,
 "registered": 256.0,
 "hour": 13.0,
 "day_of_week": 1.0,
 "month": 9.0,
 "is_clear_weather": 1.0,
 "is_rainy_weather": 0.0,
 "is_holiday_workingday": 0.0}